In [1]:
import pandas as pd
import requests
import json
import csv
import os
import time

In [2]:
url = "http://localhost:11434/api/generate"

modelo = "qwen3:4b"

In [3]:
provas = pd.read_csv("C:/Users/anara/Desktop/Pesquisa - Tutoria/feedback-open-source/dataset-biologia.csv", sep=None, engine="python")

provas.head()

,﻿question_id,answer_text,grade,question,concept,reference_answer
0,36,As células animais são todas aquelas que compõ...,3,Qual a diferença entre a célula animal e a cél...,membrana plásmática,A célula animal e vegetal apresentam formato d...
1,36,Que o nucleo de um é visível e o outro não.\n\...,1,Qual a diferença entre a célula animal e a cél...,membrana plásmática,A célula animal e vegetal apresentam formato d...
2,36,Nas vegetais tem na célula a membrana plasmáti...,2,Qual a diferença entre a célula animal e a cél...,membrana plásmática,A célula animal e vegetal apresentam formato d...
3,36,A célula vegetal possui parede celular,3,Qual a diferença entre a célula animal e a cél...,membrana plásmática,A célula animal e vegetal apresentam formato d...
4,36,uma não possui paredes e a outra possui,2,Qual a diferença entre a célula animal e a cél...,membrana plásmática,A célula animal e vegetal apresentam formato d...


In [4]:
provas_sample = provas.sample(50, random_state=123).reset_index(drop=True)

print("Total de linhas na amostra:", len(provas_sample))
print("Conceitos encontrados:")
print(provas_sample["concept"].unique())

Total de linhas na amostra: 50
Conceitos encontrados:
['filtro' 'genética' 'células justapostas' 'sistema nervoso' 'metabolismo'
 'epidídimo' 'características físicas' 'membrana plásmática' 'reação'
 'oxigênio' 'contração diafragma' 'nutrição' 'leucócitos']


In [5]:
base_prompt = """Avalie as respostas abaixo de acordo com a resposta de referência de resposta fornecida para feedback, utilizando os critérios abaixo:

1. Demonstrar domínio da modalidade escrita formal da língua portuguesa;
2. Proximidade da resposta com a resposta de referência; 

Use uma linguagem clara, respeitosa e não genérica com base no que está sendo determinado como referência de resposta para este feedback para que ele seja claro e didático. Destaque o que o aluno acertou e como ele poderia melhorar a resposta para alcançar o valor máximo da taxa de acerto.
Não invente informações.
O texto deve estar limitado a 2 sentenças de no máximo máximo 20 palavras cada sentença.
Pense nas etapas passo a passo e justifique claramente suas escolhas.

CONCEITO: 

{concept}

----

PERGUNTA: 

{question}

----

RESPOSTA DO ALUNO: 

{answer_text}

----

RESPOSTA IDEAL (referência): 

{reference_answer}

----

Retorne apenas um JSON válido no schema solicitado.

"""

In [5]:
nicol_prompt = """Avalie as respostas abaixo de acordo com a resposta de referência fornecida para feedback. Ofereça um feedback objetivo e claro levando em consideração as seguintes boas práticas de feedback propostas por Nicol & McFarlane-Dick (2006):

- Esclarecer as metas e critérios: explicação clara do que se espera em cada competência.
- Promover autoavaliação: mostrar onde o texto se aproxima ou se distancia dos critérios;
- Fornecer um feedback construtivo: apontar pontos fortes/fracos e ações de melhoria.
- Incentivar autoestima: Uso de linguagem encorajadora, reconhecendo conquistas.
- Reduzir a lacuna do desempenho: sugestões práticas de melhoria para cada competência.

Use uma linguagem clara, respeitosa e não genérica com base no que está sendo determinado como referência de resposta para este feedback, utilizando os pontos enumerados acima como boas práticas.
Não invente informações.
O texto deve estar limitado a 2 sentenças de no máximo máximo 20 palavras cada sentença.

Em seguida, pense nas etapas passo a passo e justifique claramente suas escolhas na analise do feedback.



CONCEITO: 

{concept}

----

PERGUNTA: 

{question}

----

RESPOSTA DO ALUNO: 

{answer_text}

----

RESPOSTA IDEAL (referência): 

{reference_answer}

----

A sua saída DEVE ser um objeto JSON válido, sem nenhum texto ou explicação adicional fora do JSON.
"""

In [9]:
hattie_prompt = """Avalie as respostas abaixo de acordo com a resposta de referência de resposta fornecida e forneça um feedback estruturado com base no modelo de Hattie & Timperley:

- Feed-up: Onde estou indo? O que a competência exige? (objetivo claro);
- Feedback: Como estou indo? O que foi apresentado em texto? (diagnóstico atual);
- Feed-foward: Onde ir depois? O que o aluno pode fazer para melhorar? (próximos passos)

O texto não precisa estar dividido nos 3 pontos acima, mas os 3 pontos precisam estar contemplados no feedback gerado.
Use uma linguagem clara, respeitosa e não genérica com base no que está sendo determinado como referência de resposta para este feedback.
Não invente informações.
O texto deve estar limitado a 2 sentenças de no máximo máximo 20 palavras cada sentença.



CONCEITO: 

{concept}

----

PERGUNTA: 

{question}

----

RESPOSTA DO ALUNO: 

{answer_text}

----

RESPOSTA IDEAL (referência): 

{reference_answer}

----

A sua saída DEVE ser um objeto JSON válido, sem nenhum texto ou explicação adicional fora do JSON. O objeto JSON deve seguir estritamente o seguinte esquema:

{{
    "feedback": "Texto do feedback estruturado."
}}
"""

In [10]:
nome_arquivo_csv = "qwen3_prova-biologia-results-50-hattie.csv"

In [11]:
schema_feedback = {
    "type": "object",
    "properties": {
        "feedback": {
            "type": "string"
        }
    },
    "required": ["feedback"],
    "additionalProperties": False
}


cabecalhos = [
    "question",
    "answer_text",
    "reference_answer",
    "concept",
    "feedback"
]

In [12]:
arquivo_existe = os.path.isfile(nome_arquivo_csv)

def gerar_feedback_qwen3(prompt):
    payload = {
        "model": modelo,
        "prompt": prompt,
        "stream": False,
        "think": False,
        "format": schema_feedback,
        "options": {
            "temperature": 0.2,
            "num_predict": 150,
            "seed" : 123
        }
    }

    response = requests.post(
        url,
        json=payload,
        timeout=180
    )

    response.raise_for_status()

    resposta_json = response.json()
    resposta_texto = resposta_json.get("response", "").strip()
    pensamento = resposta_json.get("thinking", "").strip()

    if not resposta_texto:
        print("Resposta vazia retornada pelo modelo.")
        if pensamento:
            print("Pensamento do modelo:")
            print(pensamento[:500])  # Exibe os primeiros 500 caracteres do pensamento 
            return ""

    try:
        dados = json.loads(resposta_texto)
        return dados.get("feedback", "").strip()

    except json.JSONDecodeError:
        print("JSON inválido retornado pelo modelo:")
        print(repr(resposta_texto))
        return ""


with open(nome_arquivo_csv, "a", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=cabecalhos)

    if not arquivo_existe:
        escritor.writeheader()

    for i, linha in provas_sample.iterrows():
        print(f"Processando resposta {i+1}/{len(provas_sample)}")

        prompt = hattie_prompt.format(
            concept=linha["concept"],
            question=linha["question"],
            answer_text=linha["answer_text"],
            reference_answer=linha["reference_answer"]
        )

        try:
            feedback = gerar_feedback_qwen3(prompt)

        except Exception as e:
            print(f"Erro ao processar linha {i+1}: {e}")
            feedback = ""

        linha_dados = {
            "question": linha["question"],
            "answer_text": linha["answer_text"],
            "reference_answer": linha["reference_answer"],
            "concept": linha["concept"],
            "feedback": feedback
        }

        escritor.writerow(linha_dados)
        f.flush()

        print(f"✅ Feedback salvo ({i+1}/{len(provas_sample)})")

        
        time.sleep(1)

print("Processamento finalizado.")
print(f"Arquivo salvo em: {nome_arquivo_csv}")

Processando resposta 1/50
✅ Feedback salvo (1/50)
Processando resposta 2/50
✅ Feedback salvo (2/50)
Processando resposta 3/50
✅ Feedback salvo (3/50)
Processando resposta 4/50
✅ Feedback salvo (4/50)
Processando resposta 5/50
✅ Feedback salvo (5/50)
Processando resposta 6/50
✅ Feedback salvo (6/50)
Processando resposta 7/50
✅ Feedback salvo (7/50)
Processando resposta 8/50
✅ Feedback salvo (8/50)
Processando resposta 9/50
✅ Feedback salvo (9/50)
Processando resposta 10/50
✅ Feedback salvo (10/50)
Processando resposta 11/50
✅ Feedback salvo (11/50)
Processando resposta 12/50
✅ Feedback salvo (12/50)
Processando resposta 13/50
✅ Feedback salvo (13/50)
Processando resposta 14/50
✅ Feedback salvo (14/50)
Processando resposta 15/50
✅ Feedback salvo (15/50)
Processando resposta 16/50
✅ Feedback salvo (16/50)
Processando resposta 17/50
✅ Feedback salvo (17/50)
Processando resposta 18/50
✅ Feedback salvo (18/50)
Processando resposta 19/50
✅ Feedback salvo (19/50)
Processando resposta 20/50
✅ F